pai_nosso_refatorado.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/alanabdmorais/pai_nosso_refatorado/blob/main/pai_nosso_refatorado.ipynb

# 🎬 Pipeline Multilíngue — Orações com Legendas Morfológicas

**Estratégia de classificação intermediária:**
O Groq gera as classificações → você revisa com uma IA → pipeline continua com os JSONs corrigidos.

---

### Fluxo completo

| # | Fase | O que faz | Ação |
|---|------|-----------|------|
| 0 | Setup | Instala deps, monta Drive, importa módulos | Automático |
| Init | — | Cria config, groq e pipeline | Automático |
| B0 | YouTube | Baixa legendas de referência (opcional) | Manual |
| 1 | Áudio | Edge TTS → .wav no Drive | Automático |
| 2 | Whisper | Transcrição → SRT bruto | Automático |
| 3 | Correção PT | Groq corrige o SRT | Automático |
| 4 | Traduções | Groq gera EN/ES/FR | Automático |
| **5A** | **Classificação** | **Groq classifica + gera pacote de revisão → pausa** | **Automático → pausa** |
| **5B** | **Revisão** | **Você corrige JSONs com IA externa** | **👤 Manual** |
| **5C** | **Recarregar** | **Pipeline carrega JSONs corrigidos** | **Automático** |
| 6 | Clipes | Baixa e corta vídeos do Pixabay | Automático |
| 7 | Vídeo base | Crédito + narração + trilha | Automático |
| 8 | Legendas ASS | Queima legendas coloridas no vídeo | Automático |

> **Retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup (rode uma vez por sessão)    ║
# ╚══════════════════════════════════════════════════╝

# Sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp nest_asyncio
print('✅ pacotes Python')

# Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
print('✅ Drive montado')

# Copiar módulos do Drive para /content/pipeline
import shutil, os, sys, logging

PASTA_MODULOS = '/content/drive/MyDrive/pipeline/modulos'  # <-- ajuste se necessário
DESTINO       = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')
else:
    print(f'⚠️  Pasta não encontrada: {PASTA_MODULOS}')
    print('   Ajuste PASTA_MODULOS acima.')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('✅ Pronto.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  INICIALIZAÇÃO — rode após o Setup                              ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# ── Ajuste aqui para trocar a oração ─────────────────────────────────────────
config = PipelineConfig(
    NOME_ORACAO  = 'pai_nosso',
    # Para outra oração:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',
)

groq     = GroqClient(api_key=userdata.get('GROQ_KEY'), nome_oracao=config.NOME_ORACAO)
pipeline = VideoPipeline(config, groq)

cp = Checkpoint()
print(config.resumo())
print()
print(cp.resumo())
print(f'\n▶  Próxima fase: {cp.proxima_fase_pendente() or "(tudo concluído)"}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAÇÃO DA ESTRUTURA DO DRIVE                           ║
# ║  (execute após a Inicialização para conferir o Drive)           ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 60)
print("📁 VERIFICANDO ESTRUTURA DO DRIVE")
print("═" * 60)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')
pasta_brutos    = Path(f'/content/drive/MyDrive/pipeline/brutos/{config.NOME_ORACAO}')

for rotulo, pasta in [("correcoes", pasta_correcoes), ("brutos", pasta_brutos)]:
    print(f"\n📁 {pasta}")
    if pasta.exists():
        arquivos = list(pasta.iterdir())
        if arquivos:
            for f in sorted(arquivos):
                print(f"   ✅ {f.name}  ({f.stat().st_size/1024:.1f} KB)")
        else:
            print("   (pasta vazia)")
    else:
        print("   ⚠️  Ainda não criada (será criada automaticamente na FASE 5A)")

print("\n" + "═" * 60)
print("💡 ESTRUTURA ESPERADA APÓS A FASE 5A:")
print(f"   MyDrive/pipeline/brutos/{config.NOME_ORACAO}/")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print(f"   MyDrive/pipeline/correcoes/{config.NOME_ORACAO}/")
print(f"   ├── prompt_revisao.md")
print(f"   ├── relatorio_classificacoes.csv")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print("═" * 60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA DO PIPELINE                                ║
# ╚══════════════════════════════════════════════════════════════════╝

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
import shutil, sys

output_limpeza = widgets.Output()

def limpeza_rapida(tipo):
    with output_limpeza:
        clear_output(wait=True)
        print("═" * 65)
        print(f"🧹 LIMPANDO: {tipo}")
        print("═" * 65)
        cont = 0

        if tipo in ('audio', 'tudo'):
            print("\n🎵 Removendo áudios...")
            for f in Path('.').glob('*_audio.wav'):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

        if tipo in ('legendas', 'tudo'):
            print("\n📝 Removendo legendas...")
            for f in Path('.').glob('*.srt'):
                if 'yt_ref' not in f.name:
                    f.unlink(); cont += 1; print(f"   🗑️ {f.name}")
            for f in Path('.').glob('*.ass'):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

        if tipo in ('videos', 'tudo'):
            print("\n🎬 Removendo vídeos...")
            for pat in ['*_base.mp4', '*_final.mp4', 'clipe_*.mp4', 'temp_*.mp4']:
                for f in Path('.').glob(pat):
                    f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

        if tipo in ('jsons', 'tudo'):
            print("\n📊 Removendo JSONs locais...")
            for f in Path('.').glob('classificacao_*.json'):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

        if tipo in ('checkpoint', 'tudo'):
            print("\n📌 Removendo checkpoint...")
            cp = Path('checkpoint.json')
            if cp.exists():
                cp.unlink(); cont += 1; print("   🗑️ checkpoint.json")

        if tipo in ('pastas', 'tudo'):
            print("\n📁 Removendo pastas temporárias...")
            for pasta in ['clipes_cortados', 'clipes_prontos', 'temp_raw', '__pycache__']:
                p = Path(pasta)
                if p.exists():
                    shutil.rmtree(p); cont += 1; print(f"   🗑️ {pasta}/")

        if tipo in ('cache', 'tudo'):
            print("\n📦 Limpando cache Python...")
            mods = ['groq_client', 'video_pipeline', 'config', 'classification',
                    'checkpoint', 'drive_utils', 'ffmpeg_utils', 'srt_utils',
                    'models', 'constants']
            for m in mods:
                if m in sys.modules:
                    del sys.modules[m]; cont += 1; print(f"   🗑️ {m}")

        if tipo in ('drive', 'tudo_drive'):
            print("\n☁️ Removendo JSONs do Drive (correcoes/)...")
            drive_correcoes = Path('/content/drive/MyDrive/pipeline/correcoes')
            if drive_correcoes.exists():
                for d in drive_correcoes.iterdir():
                    if d.is_dir():
                        for f in d.glob('classificacao_*.json'):
                            f.unlink(); cont += 1; print(f"   🗑️ Drive: {d.name}/{f.name}")

        print("\n" + "═" * 65)
        print(f"✅ Limpeza concluída! {cont} item(ns) removido(s)")
        print("═" * 65)

btn_audio      = widgets.Button(description='🎵 Áudios',          button_style='warning', layout=widgets.Layout(width='130px'))
btn_legendas   = widgets.Button(description='📝 Legendas',        button_style='warning', layout=widgets.Layout(width='130px'))
btn_videos     = widgets.Button(description='🎬 Vídeos',          button_style='warning', layout=widgets.Layout(width='130px'))
btn_jsons      = widgets.Button(description='📊 JSONs (local)',   button_style='warning', layout=widgets.Layout(width='140px'))
btn_checkpoint = widgets.Button(description='📌 Checkpoint',      button_style='warning', layout=widgets.Layout(width='130px'))
btn_pastas     = widgets.Button(description='📁 Pastas temp',     button_style='warning', layout=widgets.Layout(width='130px'))
btn_cache      = widgets.Button(description='📦 Cache Python',    button_style='warning', layout=widgets.Layout(width='140px'))
btn_drive      = widgets.Button(description='☁️ JSONs no Drive',  button_style='danger',  layout=widgets.Layout(width='150px'))
btn_tudo       = widgets.Button(description='🔥 LIMPAR TUDO (local)',    button_style='danger', layout=widgets.Layout(width='180px', height='45px'))
btn_tudo_drive = widgets.Button(description='🔥🔥 LIMPAR TUDO + DRIVE', button_style='danger', layout=widgets.Layout(width='220px', height='45px'))

btn_audio.on_click(lambda b: limpeza_rapida('audio'))
btn_legendas.on_click(lambda b: limpeza_rapida('legendas'))
btn_videos.on_click(lambda b: limpeza_rapida('videos'))
btn_jsons.on_click(lambda b: limpeza_rapida('jsons'))
btn_checkpoint.on_click(lambda b: limpeza_rapida('checkpoint'))
btn_pastas.on_click(lambda b: limpeza_rapida('pastas'))
btn_cache.on_click(lambda b: limpeza_rapida('cache'))
btn_drive.on_click(lambda b: limpeza_rapida('drive'))
btn_tudo.on_click(lambda b: limpeza_rapida('tudo'))
btn_tudo_drive.on_click(lambda b: limpeza_rapida('tudo_drive'))

print("═" * 65)
print("🎯 LIMPEZA SELETIVA")
print("═" * 65)
display(HTML("""
<table style="width:100%; font-size:12px;">
 <tr><th style="text-align:left">Botão</th><th style="text-align:left">O que limpa</th></tr>
 <tr><td>🎵 Áudios</td><td>Arquivos .wav e .mp3 gerados</td></tr>
 <tr><td>📝 Legendas</td><td>Arquivos .srt e .ass (exceto yt_ref)</td></tr>
 <tr><td>🎬 Vídeos</td><td>Vídeos gerados (*_base.mp4, *_final.mp4, clipes)</td></tr>
 <tr><td>📊 JSONs (local)</td><td>JSONs de classificação na pasta /content/</td></tr>
 <tr><td>📌 Checkpoint</td><td>checkpoint.json (progresso do pipeline)</td></tr>
 <tr><td>📁 Pastas temp</td><td>Pastas temporárias (clipes_cortados, etc.)</td></tr>
 <tr><td>📦 Cache Python</td><td>Módulos carregados na memória</td></tr>
 <tr><td>☁️ JSONs no Drive</td><td>JSONs corrigidos em MyDrive/pipeline/correcoes/</td></tr>
 <tr><td>🔥 LIMPAR TUDO</td><td>Todos os itens acima (exceto Drive)</td></tr>
 <tr><td>🔥🔥 LIMPAR TUDO + DRIVE</td><td>TUDO, incluindo JSONs corrigidos no Drive</td></tr>
</table>
"""))
display(widgets.HBox([btn_audio, btn_legendas, btn_videos, btn_jsons]))
display(widgets.HBox([btn_checkpoint, btn_pastas, btn_cache, btn_drive]))
display(widgets.HBox([btn_tudo, btn_tudo_drive]))
display(output_limpeza)

### 🔵 Célula B0 — Opcional: baixar legendas do YouTube
Execute **antes** das fases 1–8 para ter traduções mais fiéis. Se pular, o Groq traduz a partir do PT.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — Baixar legendas do YouTube                        ║
# ║  (BAIXA TAMBÉM A REFERÊNCIA PT para a FASE 3)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
import shutil
from drive_utils import DriveClient

cfg   = config
drive = DriveClient.get()

drive.download_se_ausente(cfg.ID_PASTA_COOKIES, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

print("═" * 60)
print("📺 BAIXANDO LEGENDAS DO YOUTUBE")
print("═" * 60)

# ── COLOQUE AS URLs DOS VÍDEOS COM LEGENDAS ABAIXO ────────────────────────────
URLS = {
    'pt': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'en': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'es': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'fr': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
}

for lang, url in URLS.items():
    nome_srt = cfg.nome_srt(lang)
    print(f'\n⬇️  Baixando {lang.upper()} de: {url[:50]}...')
    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang, '--write-auto-sub',
        '--skip-download', '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}', *cookies_flag, url
    ]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    encontrado = False
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt} salvo')
        encontrado = True
        break
    if not encontrado:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')

# Criar referência PT para a FASE 3
print("\n" + "═" * 60)
print("📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)")
print("═" * 60)
srt_pt   = Path(cfg.nome_srt('pt'))
ref_path = Path(f'{cfg.NOME_ORACAO}_yt_ref_pt.srt')
if srt_pt.exists():
    shutil.copy(srt_pt, ref_path)
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f'✅ Referência PT salva: {ref_path}  ({len(ref_legendas)} segmentos)')
    print(f'   Preview: {ref_legendas[0].texto[:80]}')
else:
    print(f'⚠️  SRT PT não encontrado. A FASE 3 corrigirá sem referência.')
print("═" * 60)

### ▶ Fases 1–4 — Automáticas

In [ ]:
# ── FASE 1: Áudio com Edge TTS ───────────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')

# ── FASE 2: Transcrição Whisper ──────────────────────────────────────────────
srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
print(f'{srt_bruto.name}:')
for leg in ler_srt(srt_bruto)[:4]:
    print(f'  [{leg.inicio_str}]  {leg.texto}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 3 — Correção PT com Groq                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import time, re
from datetime import datetime

ref_path = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
print("═" * 60)
if ref_path.exists():
    from srt_utils import ler_srt as _ler
    ref_legs = _ler(ref_path)
    print(f"✅ Referência PT encontrada: {len(ref_legs)} segmentos")
else:
    print(f"⚠️  Referência PT não encontrada — corrigindo sem referência")
print("═" * 60)

def corrigir_com_retry(max_tentativas=5):
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"\n🔄 Tentativa {tentativa}/{max_tentativas} - {datetime.now().strftime('%H:%M:%S')}")
            return pipeline.fase3_corrigir_pt()
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate limit' in msg.lower():
                m = re.search(r'(\d+)m(\d+\.?\d*)s', msg)
                espera = int(m.group(1)) * 60 + float(m.group(2)) if m else 30
                print(f"\n⏳ Rate limit — aguardando {espera:.0f}s...")
                for i in range(int(espera), 0, -5):
                    print(f"   ⏰ {i}s...", end='\r')
                    time.sleep(5)
                continue
            raise
    raise Exception(f"Falha após {max_tentativas} tentativas")

try:
    legendas_pt = corrigir_com_retry()
    print(f"\n✅ {len(legendas_pt)} legendas PT corrigidas")
    for leg in legendas_pt:
        print(f'  #{leg.id:02d}  {leg.texto}')
except Exception as e:
    print(f'\n❌ Erro na correção: {e}')

# ── FASE 4: Traduções EN/ES/FR ───────────────────────────────────────────────
from srt_utils import ler_srt
if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)

pipeline.legendas_idiomas = pipeline.fase4_traduzir(legendas_pt)

print('Primeiras 2 legendas por idioma:')
for i in range(min(2, len(legendas_pt))):
    for lang in config.IDIOMAS:
        legs = pipeline.legendas_idiomas.get(lang, [])
        if i < len(legs):
            print(f'  {lang.upper()}: {legs[i].texto}')
    print()

### ▶ Fase 5A — Classificação morfológica (Groq) + Pacote de revisão

O pipeline vai:
1. Classificar todos os idiomas via Groq (forçando do zero)
2. Salvar os JSONs brutos em `brutos/` no Drive
3. **Gerar o pacote de revisão** completo em `correcoes/` no Drive:
   - `prompt_revisao.md` — instruções para a IA revisora
   - `relatorio_classificacoes.csv` — todas as palavras com coluna `Sugestao`
   - Os 4 JSONs brutos prontos para correção
4. **Pausar** e mostrar instruções para a Fase 5B

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5A — Classificação Groq + Geração do Pacote de Revisão   ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt
import time, re
from datetime import datetime

if 'legendas_pt' not in dir() or not legendas_pt:
    legendas_pt = ler_srt(config.NOME_SRT_PT)
if not pipeline.legendas_idiomas:
    pipeline.legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)

print("═" * 65)
print("🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA + PACOTE DE REVISÃO")
print("═" * 65)

def classificar_com_retry(max_tentativas=5):
    for tentativa in range(1, max_tentativas + 1):
        try:
            print(f"\n🚀 Tentativa {tentativa}/{max_tentativas} - {datetime.now().strftime('%H:%M:%S')}")
            for lang in config.IDIOMAS:
                print(f"\n🔤 Classificando {lang.upper()}...")
                for leg in pipeline.legendas_idiomas[lang]:
                    leg.palavras = []
                pipeline._clf.classificar_idioma(pipeline.legendas_idiomas[lang], lang, forcar=True)
                print(f"   ✅ {lang.upper()} classificado")
                if lang != config.IDIOMAS[-1]:
                    print("   ⏳ Aguardando 5s...")
                    time.sleep(5)
            pipeline._cp.salvar("classificacoes_feitas", {"fonte": "groq_forcado"})
            return pipeline.legendas_idiomas
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate limit' in msg.lower():
                m = re.search(r'(\d+)m(\d+\.?\d*)s', msg)
                espera = int(m.group(1)) * 60 + float(m.group(2)) if m else 30
                print(f"\n⏳ Rate limit — aguardando {espera:.0f}s...")
                for i in range(int(espera), 0, -5):
                    print(f"   ⏰ {i}s...", end='\r')
                    time.sleep(5)
                continue
            raise
    raise Exception(f"Falha após {max_tentativas} tentativas")

try:
    pipeline.legendas_idiomas = classificar_com_retry()

    print("\n" + "═" * 65)
    print("✅ CLASSIFICAÇÃO CONCLUÍDA — gerando pacote de revisão...")
    print("═" * 65)

    # ── Gera pacote completo (JSONs + CSV com Sugestao + prompt_revisao.md) ──
    pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)

    print(f"\n📦 Pacote gerado em: {pacote}")
    print("\n📁 Arquivos disponíveis em:")
    print(f"   MyDrive/pipeline/correcoes/{config.NOME_ORACAO}/")
    print(f"   ├── prompt_revisao.md          ← cole numa IA junto com os JSONs")
    print(f"   ├── relatorio_classificacoes.csv ← mostra palavras inválidas + sugestões")
    print(f"   ├── classificacao_{config.NOME_ORACAO}_pt.json")
    print(f"   ├── classificacao_{config.NOME_ORACAO}_en.json")
    print(f"   ├── classificacao_{config.NOME_ORACAO}_es.json")
    print(f"   └── classificacao_{config.NOME_ORACAO}_fr.json")

    print("\n" + "═" * 65)
    print("⏸️  PIPELINE PAUSADO — Fase 5B (revisão manual)")
    print("═" * 65)
    print("\n📋 PRÓXIMOS PASSOS:")
    print("   1. Acesse: MyDrive/pipeline/correcoes/{}/".format(config.NOME_ORACAO))
    print("   2. Abra prompt_revisao.md e copie o conteúdo")
    print("   3. Cole numa IA (Claude, GPT, etc.) junto com os 4 JSONs")
    print("   4. A IA retorna os JSONs corrigidos")
    print("   5. Salve os JSONs corrigidos na mesma pasta do Drive")
    print("   6. Execute a FASE 5C abaixo")
    print("═" * 65)

except Exception as e:
    print(f'\n❌ Erro: {e}')

### ⏸️ Fase 5B — Revisão manual pela IA ← **Você faz isso**

Após a Fase 5A concluir:

1. **Abra o Google Drive** → `MyDrive/pipeline/correcoes/pai_nosso/`
2. Você verá:
   - `prompt_revisao.md` ← **copie e cole numa IA junto com os 4 JSONs**
   - `relatorio_classificacoes.csv` ← consulte a coluna `Sugestao` para ver correções sugeridas
   - Os 4 JSONs brutos para corrigir
3. A IA retorna os JSONs corrigidos
4. **Salve os JSONs corrigidos na mesma pasta** (substituindo os originais)
5. Execute a Fase 5C abaixo

---
### ▶ Fase 5C — Recarregar classificações corrigidas

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5C — Recarregar do Drive após revisão                    ║
# ║  Execute APÓS salvar os JSONs corrigidos no Drive              ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📥 FASE 5C — RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS")
print("═" * 65)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')

if not pasta_correcoes.exists():
    print(f"❌ Pasta não encontrada: {pasta_correcoes}")
    print("   Execute a FASE 5A primeiro.")
else:
    jsons_ok = []
    jsons_faltando = []
    for lang in config.IDIOMAS:
        p = pasta_correcoes / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
        if p.exists():
            jsons_ok.append(lang.upper())
            print(f"   ✅ {p.name}")
        else:
            jsons_faltando.append(lang.upper())
            print(f"   ❌ {p.name}")

    if len(jsons_ok) == 4:
        print("\n✅ Todos os 4 JSONs encontrados! Carregando...")
        pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

        print("\n📋 Preview das classificações:")
        print("─" * 55)
        for lang in config.IDIOMAS:
            legs = pipeline.legendas_idiomas.get(lang, [])
            if legs and legs[0].palavras:
                leg = legs[0]
                print(f'\n  {lang.upper()} — "{leg.texto[:50]}"')
                from constants import CORES_HTML
                for p in leg.palavras[:4]:
                    ok  = '✅' if p.classe in CORES_HTML else '❌'
                    print(f'    {ok} {p.texto:<18} {p.classe}')
                if len(leg.palavras) > 4:
                    print(f'       ... (+{len(leg.palavras)-4} palavras)')

        print("\n" + "═" * 65)
        print("🎬 Pronto — execute as Fases 6, 7 e 8")
        print("═" * 65)
    else:
        print(f"\n⚠️  Faltam: {', '.join(jsons_faltando)}")
        print("   Coloque os JSONs corrigidos na pasta e execute novamente.")

### ▶ Fases 6–8 — Vídeo final (rode após a Fase 5C)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASES 6, 7 e 8 — Vídeo final                 ║
# ╚══════════════════════════════════════════════════╝

video_final = pipeline.continuar()

if video_final:
    print(f'\n🎬 Vídeo final: {video_final}')
    print(f'   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB')
    print(f'   Salvo no Drive: pasta Vídeos')

### 🚀 RUN ALL — Pipeline completo com checkpoint

Use quando quiser rodar tudo de uma vez. O pipeline pausa automaticamente na Fase 5A
se houver classificações a revisar. Após a revisão, rode a célula da Fase 5C
e depois a célula das Fases 6–8.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  RUN ALL                                       ║
# ╚══════════════════════════════════════════════════╝
from video_pipeline import ClassificacaoPendenteError

resultado = pipeline.run(
    # from_phase='clipes_cortados'  # descomente para retomar de uma fase
)

if resultado is None:
    print('\n⏸️  Pipeline pausado na Fase 5A.')
    print('   Siga as instruções acima, depois execute a célula FASE 5C.')
else:
    print(f'\n🎉 Vídeo final: {resultado}')
    print(pipeline._cp.resumo())

### 🔧 Utilitários

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  UTILITÁRIOS — descomente o que precisar       ║
# ╚══════════════════════════════════════════════════╝
from checkpoint import Checkpoint
cp = Checkpoint()
print(cp.resumo())
print()
pipeline._clf.imprimir_status()

# ── Checkpoint ────────────────────────────────────────────────────────────────
# cp.resetar_tudo()
# cp.reiniciar_de('classificacoes_feitas')

# ── Classificações ────────────────────────────────────────────────────────────
# pipeline._clf.imprimir_status()
# pipeline._clf.classificar_idioma(pipeline.legendas_idiomas['en'], 'en', forcar=True)

# Gerar pacote de revisão manualmente (sem reclassificar):
# pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)
# print(f"📦 Pacote: {pacote}")

# Verificar classes inválidas:
# from constants import CORES_HTML
# for lang, legendas in pipeline.legendas_idiomas.items():
#     for leg in legendas:
#         for p in leg.palavras:
#             if p.classe not in CORES_HTML:
#                 print(f'  [{lang.upper()} leg.{leg.id}] "{p.texto}" → {p.classe} ❌')